# Stage 20 — Filtragem e seleção das épocas

Objetivo: selecionar somente os ensaios de **fala interna**, filtrar o EEG nas bandas alfa e beta e recortar o intervalo de 1,5 a 3,5 segundos.

Execute as células na ordem. A última célula é a única que processa e salva todas as sessões.


## 1. Importações

- `Path`: manipulação de caminhos;
- `pickle`: leitura da matriz de eventos;
- `mne`: leitura e filtragem do EEG;
- `numpy`: organização e salvamento dos arrays.


In [ ]:
from pathlib import Path
import pickle

import mne
import numpy as np


## 2. Caminhos e parâmetros

As bandas são guardadas em um dicionário para deixar explícito o significado de cada índice do tensor final.


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "thinking_outloud_dataset" / "derivatives"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_20_band_filtering_and_epoch"

BANDS = {
    "alpha": (8, 12),
    "beta": (12, 30),
}
START_SECONDS = 1.5
END_SECONDS = 3.5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)


## 3. Descobrir as sessões disponíveis

Em vez de fixar participantes e sessões no código, procuramos os arquivos `.fif` existentes.


In [ ]:
epoch_files = sorted(INPUT_DIR.glob("sub-*/ses-*/*_eeg-epo.fif"))

print(f"Sessões encontradas: {len(epoch_files)}")
for path in epoch_files[:3]:
    print(" -", path.relative_to(PROJECT_ROOT))


## 4. Entender os eventos de uma sessão

Cada linha corresponde a um ensaio. A coluna 1 contém a palavra (`0` a `3`) e a coluna 2 contém a condição. O valor `1` representa fala interna.


In [ ]:
example_epochs_path = epoch_files[0]
example_name = example_epochs_path.name.replace("_eeg-epo.fif", "")
example_events_path = example_epochs_path.with_name(f"{example_name}_events.dat")

with example_events_path.open("rb") as file:
    example_events = pickle.load(file)

print("Formato dos eventos:", example_events.shape)
print("Primeiras linhas:\n", example_events[:5])
print("Condições e quantidades:", np.unique(example_events[:, 2], return_counts=True))


## 5. Função que processa uma sessão

A função retorna os dados, os rótulos e informações úteis para conferência. Nenhum arquivo é salvo dentro dela.


In [ ]:
def process_session(epochs_path):
    session_name = epochs_path.name.replace("_eeg-epo.fif", "")
    events_path = epochs_path.with_name(f"{session_name}_events.dat")

    epochs = mne.read_epochs(epochs_path, preload=True, verbose=False)
    with events_path.open("rb") as file:
        events = pickle.load(file)

    inner_speech_mask = events[:, 2] == 1
    inner_epochs = epochs[inner_speech_mask]
    labels = events[inner_speech_mask, 1].astype(int)

    sampling_rate = float(inner_epochs.info["sfreq"])
    start_sample = round(START_SECONDS * sampling_rate)
    end_sample = round(END_SECONDS * sampling_rate)

    filtered_bands = []
    for band_name, (low_frequency, high_frequency) in BANDS.items():
        filtered_epochs = inner_epochs.copy().filter(
            l_freq=low_frequency,
            h_freq=high_frequency,
            picks="eeg",
            method="fir",
            phase="zero",
            fir_design="firwin",
            verbose=False,
        )
        band_data = filtered_epochs.get_data()[:, :, start_sample:end_sample]
        filtered_bands.append(band_data)

    data = np.stack(filtered_bands, axis=1)
    return session_name, data, labels, inner_epochs.ch_names, sampling_rate


## 6. Testar com uma sessão

Esta célula permite conferir dimensões e classes antes de processar tudo.


In [ ]:
name, data, labels, channel_names, sampling_rate = process_session(epoch_files[0])

print("Sessão:", name)
print("Taxa de amostragem:", sampling_rate)
print("Dados (épocas, bandas, canais, tempo):", data.shape)
print("Rótulos:", labels.shape)
print("Classes:", np.unique(labels, return_counts=True))
print("Primeiros canais:", channel_names[:5])


## 7. Processar e salvar todas as sessões

Execute esta célula somente depois de conferir o exemplo acima.


In [ ]:
for epochs_path in epoch_files:
    name, data, labels, _, _ = process_session(epochs_path)

    np.save(OUTPUT_DIR / f"{name}_inner_bands.npy", data)
    np.save(OUTPUT_DIR / f"{name}_inner_bands_labels.npy", labels)

    print(f"{name}: dados={data.shape}, rótulos={labels.shape}")

print("Stage 20 finalizada.")
